In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv")
df["attrition_flag"] = df["Attrition"].map({"No": 0, "Yes": 1})

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,attrition_flag
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,80,0,8,0,1,6,4,0,5,1
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,80,1,10,3,3,10,7,1,7,0
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,80,0,7,3,3,0,0,0,0,1
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,80,0,8,3,3,8,7,3,0,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,80,1,6,3,3,2,2,2,2,0


In [2]:
df.nunique().sort_values()

EmployeeCount                  1
Over18                         1
StandardHours                  1
Attrition                      2
OverTime                       2
PerformanceRating              2
Gender                         2
attrition_flag                 2
Department                     3
MaritalStatus                  3
BusinessTravel                 3
RelationshipSatisfaction       4
JobSatisfaction                4
EnvironmentSatisfaction        4
JobInvolvement                 4
StockOptionLevel               4
WorkLifeBalance                4
Education                      5
JobLevel                       5
EducationField                 6
TrainingTimesLastYear          7
JobRole                        9
NumCompaniesWorked            10
PercentSalaryHike             15
YearsSinceLastPromotion       16
YearsWithCurrManager          18
YearsInCurrentRole            19
DistanceFromHome              29
YearsAtCompany                37
TotalWorkingYears             40
Age       

In [3]:
drop_cols = ["EmployeeCount", "Over18", "StandardHours", "EmployeeNumber"]
df = df.drop(columns=drop_cols)

df.shape

(1470, 32)

In [4]:
# 수치형 파생변수
df["income_log"] = np.log1p(df["MonthlyIncome"])
df["tenure_ratio"] = df["YearsAtCompany"] / (df["TotalWorkingYears"] + 1)
df["promotion_gap_ratio"] = df["YearsSinceLastPromotion"] / (df["YearsAtCompany"] + 1)
df["manager_tenure_ratio"] = df["YearsWithCurrManager"] / (df["YearsAtCompany"] + 1)
df["distance_per_income"] = df["DistanceFromHome"] / (df["MonthlyIncome"] + 1)

# 이진형 파생변수
df["overtime_flag"] = df["OverTime"].map({"No": 0, "Yes": 1})
df["travel_frequently_flag"] = (df["BusinessTravel"] == "Travel_Frequently").astype(int)
df["single_flag"] = (df["MaritalStatus"] == "Single").astype(int)
df["male_flag"] = (df["Gender"] == "Male").astype(int)

# 구간형 파생변수
df["age_group"] = pd.cut(
    df["Age"],
    bins=[17, 25, 35, 45, 55, 65],
    labels=["18-25", "26-35", "36-45", "46-55", "56-65"]
)

df["income_group"] = pd.qcut(
    df["MonthlyIncome"],
    q=4,
    labels=["저소득", "중하", "중상", "고소득"]
)

df["tenure_group"] = pd.cut(
    df["YearsAtCompany"],
    bins=[-1, 2, 5, 10, 20, 40],
    labels=["0-2년", "3-5년", "6-10년", "11-20년", "20년 초과"]
)

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,promotion_gap_ratio,manager_tenure_ratio,distance_per_income,overtime_flag,travel_frequently_flag,single_flag,male_flag,age_group,income_group,tenure_group
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,0.000000,0.714286,0.000167,1,0,1,0,36-45,중상,6-10년
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,0.090909,0.636364,0.001559,0,1,0,1,46-55,중상,6-10년
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,0.000000,0.000000,0.000956,1,0,1,1,36-45,저소득,0-2년
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,0.333333,0.000000,0.001031,1,1,0,0,26-35,저소득,6-10년
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,0.666667,0.666667,0.000577,0,0,0,1,26-35,중하,0-2년


In [5]:
new_cols = [
    "income_log", "tenure_ratio", "promotion_gap_ratio", "manager_tenure_ratio",
    "distance_per_income", "overtime_flag", "travel_frequently_flag",
    "single_flag", "male_flag", "age_group", "income_group", "tenure_group"
]

df[new_cols].head()

,income_log,tenure_ratio,promotion_gap_ratio,manager_tenure_ratio,distance_per_income,overtime_flag,travel_frequently_flag,single_flag,male_flag,age_group,income_group,tenure_group
0,8.698514,0.666667,0.000000,0.714286,0.000167,1,0,1,0,36-45,중상,6-10년
1,8.543056,0.909091,0.090909,0.636364,0.001559,0,1,0,1,46-55,중상,6-10년
2,7.645398,0.000000,0.000000,0.000000,0.000956,1,0,1,1,36-45,저소득,0-2년
3,7.975908,0.888889,0.333333,0.000000,0.001031,1,1,0,0,26-35,저소득,6-10년
4,8.151622,0.285714,0.666667,0.666667,0.000577,0,0,0,1,26-35,중하,0-2년


In [6]:
df[new_cols].isnull().sum()

income_log                0
tenure_ratio              0
promotion_gap_ratio       0
manager_tenure_ratio      0
distance_per_income       0
overtime_flag             0
travel_frequently_flag    0
single_flag               0
male_flag                 0
age_group                 0
income_group              0
tenure_group              0
dtype: int64

In [7]:
for col in ["age_group", "income_group", "tenure_group"]:
    result = (
        df.groupby(col, observed=False)["attrition_flag"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
    )
    print(f"\n===== {col} =====")
    print(result)


===== age_group =====
               mean  count
age_group                 
18-25      0.357724    123
26-35      0.191419    606
56-65      0.170213     47
46-55      0.115044    226
36-45      0.091880    468

===== income_group =====
                  mean  count
income_group                 
저소득           0.292683    369
중하            0.142077    366
중상            0.106267    367
고소득           0.103261    368

===== tenure_group =====
                  mean  count
tenure_group                 
0-2년          0.298246    342
3-5년          0.138249    434
6-10년         0.122768    448
20년 초과        0.121212     66
11-20년        0.066667    180


In [8]:
for col in ["overtime_flag", "travel_frequently_flag", "single_flag"]:
    result = (
        df.groupby(col)["attrition_flag"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
    )
    print(f"\n===== {col} =====")
    print(result)


===== overtime_flag =====
                   mean  count
overtime_flag                 
1              0.305288    416
0              0.104364   1054

===== travel_frequently_flag =====
                            mean  count
travel_frequently_flag                 
1                       0.249097    277
0                       0.140821   1193

===== single_flag =====
                 mean  count
single_flag                 
1            0.255319    470
0            0.117000   1000


In [9]:
output_path = "../data/processed/hr_attrition_v1.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(output_path)

../data/processed/hr_attrition_v1.csv


1차 파생변수 설계 메모

- 원본 데이터에는 조직 이동, 리더 변경, 급여 인상 이력 등 상세한 HR 이벤트 데이터가 없음
- 따라서 이번 단계에서는 현재 데이터셋에서 구성 가능한 대리 변수를 생성함  
  - 근속 관련 변수(tenure_ratio, tenure_group)  
  - 보상 관련 변수(income_log, income_group)    
  - 근무부담 관련 변수(overtime_flag, travel_frequently_flag)    
  - 조직 정착 관련 변수(manager_tenure_ratio, promotion_gap_ratio)
- 이는 향후 이직 리스크를 개인 속성뿐 아니라 근무환경과 조직 정착도 관점에서 설명하기 위한 기반 데이터셋으로 사용